# 🏎️ LazyPredict Benchmarking
**Extended · Data Pattern Recognition for the Rest of Us**

> Before committing to a model, benchmark dozens of algorithms in one call. LazyPredict tells you which model families are worth deeper investment — in minutes instead of days.

### Dataset
**Telecom Customer Churn**
Predict which customers will cancel their service. Churn costs telecom companies billions annually — retaining customers is 5-25x cheaper than acquiring new ones. This is a classic imbalanced binary classification problem with real business stakes.

### Time: ~45 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install lazypredict -q
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
print("\u2713 Setup complete — lazypredict installed")

In [ ]:
# Telecom Churn Dataset (IBM Telco — widely used in industry)
import numpy as np, pandas as pd

try:
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    telco = pd.read_csv(url)
    telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
    telco = telco.dropna()
    telco['Churn_num'] = (telco['Churn'] == 'Yes').astype(int)
    cat_cols = telco.select_dtypes('object').columns.drop(['customerID','Churn'])
    le = LabelEncoder()
    for c in cat_cols:
        telco[c] = le.fit_transform(telco[c])
    feature_cols = [c for c in telco.columns
                    if c not in ['customerID','Churn','Churn_num']]
    X = telco[feature_cols].values
    y = telco['Churn_num'].values
    print("Loaded IBM Telco Churn dataset")
except:
    np.random.seed(42)
    n = 7043  # matches real Telco dataset size
    tenure         = np.random.randint(0, 72, n)
    monthly_charges= np.random.uniform(18, 120, n)
    total_charges  = tenure * monthly_charges * np.random.uniform(0.9,1.1,n)
    contract       = np.random.choice([0,1,2], n, p=[0.55,0.24,0.21])  # M2M/1yr/2yr
    internet_service=np.random.choice([0,1,2], n, p=[0.21,0.44,0.35])  # No/DSL/Fiber
    tech_support   = np.random.binomial(1,0.28,n)
    paperless      = np.random.binomial(1,0.59,n)
    num_services   = np.random.randint(0,7,n)
    senior         = np.random.binomial(1,0.16,n)

    log_odds = (-1.5
                - 0.05*tenure
                + 0.015*monthly_charges
                + 1.2*(contract==0).astype(float)
                - 0.8*(contract==2).astype(float)
                + 0.4*(internet_service==2).astype(float)
                - 0.3*tech_support
                + 0.2*senior)
    prob_churn = 1/(1+np.exp(-log_odds))
    churn = (np.random.uniform(0,1,n) < prob_churn).astype(int)

    X = np.column_stack([tenure,monthly_charges,total_charges,contract,
                         internet_service,tech_support,paperless,
                         num_services,senior])
    y = churn
    feature_cols = ['tenure','monthly_charges','total_charges','contract',
                    'internet_service','tech_support','paperless',
                    'num_services','senior']
    print("Using synthetic Telco-like churn dataset")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)
print(f"\nTelecom Churn dataset: {len(y):,} customers")
print(f"Churn rate: {y.mean():.1%}")
print(f"Training: {len(y_tr):,}  |  Test: {len(y_te):,}")
print("\nRunning LazyPredict on all sklearn classifiers...")
print("(This benchmarks 30+ models — may take 2-3 minutes)")

In [ ]:
# Run LazyPredict — benchmark all classifiers
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions = clf.fit(X_tr, X_te, y_tr, y_te)

print("=== LazyPredict Results — Telecom Churn (sorted by AUC-ROC) ===\n")
# Show top 15
display_cols = [c for c in ['Accuracy','Balanced Accuracy','ROC AUC','F1 Score','Time Taken']
                if c in models_df.columns]
if 'ROC AUC' in models_df.columns:
    models_sorted = models_df.sort_values('ROC AUC', ascending=False)
else:
    models_sorted = models_df.sort_values('Accuracy', ascending=False)
print(models_sorted.head(15)[display_cols].to_string())

In [ ]:
# Visualize the benchmark results
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# AUC-ROC comparison
metric = 'ROC AUC' if 'ROC AUC' in models_df.columns else 'Accuracy'
top_models = models_sorted.head(15)
colors_bar = ['#1a7a45' if i < 3 else '#1e5fa8' if i < 8 else '#888'
              for i in range(len(top_models))]
axes[0].barh(range(len(top_models)), top_models[metric].values[::-1],
             color=colors_bar[::-1], edgecolor='white')
axes[0].set_yticks(range(len(top_models)))
axes[0].set_yticklabels([n[:25] for n in top_models.index[::-1]], fontsize=9)
axes[0].set_xlabel(metric)
axes[0].set_title(f'LazyPredict: {metric} by Model\n'
                  f'Green = top 3, Blue = top 8')
axes[0].axvline(top_models[metric].median(), color='#e85d20', lw=1.5, ls='--',
               label=f'Median = {top_models[metric].median():.3f}')
axes[0].legend()

# Speed vs accuracy tradeoff
if 'Time Taken' in models_df.columns:
    axes[1].scatter(models_df['Time Taken'], models_df[metric],
                    color='#1e5fa8', alpha=0.7, s=40)
    for i, (name, row) in enumerate(models_df.iterrows()):
        if row[metric] > models_df[metric].quantile(0.85):
            axes[1].annotate(name[:15], (row['Time Taken'], row[metric]),
                            fontsize=7, alpha=0.8)
    axes[1].set_xlabel('Training Time (seconds)')
    axes[1].set_ylabel(metric)
    axes[1].set_title('Speed vs Performance Tradeoff\n'
                      'Best = top-left (fast AND accurate)')

plt.tight_layout(); plt.show()
best_model = models_sorted.index[0]
print(f"\n\u2714 LazyPredict winner: {best_model}")
print(f"   This is a STARTING POINT — now tune this model with proper cross-validation")
print(f"   LazyPredict does NOT replace rigorous model evaluation!")

In [ ]:
# Business framing: what does AUC mean for churn?
print("=== Business Interpretation ===\n")
print("A churn model with AUC=0.85 means:")
print("  If you randomly pick 1 churner and 1 non-churner,")
print("  the model ranks the churner higher 85% of the time.")
print()
print("At a retention budget of $50/customer and 7,043 customers:")
churn_rate = y.mean()
n_churners = int(len(y) * churn_rate)
budget_per_customer = 50
print(f"  Total churners to prevent: ~{n_churners}")
print(f"  If model catches 70% of churners at a 2:1 false positive rate:")
caught = int(n_churners * 0.70)
contacted = caught * 3  # 2:1 FP ratio
cost = contacted * budget_per_customer
prevented_revenue = caught * 120 * 12  # $120/mo avg, 1 year
print(f"  Customers contacted: {contacted:,}")
print(f"  Campaign cost: ${cost:,}")
print(f"  Revenue retained (est.): ${prevented_revenue:,}")
print(f"  Net ROI: {(prevented_revenue - cost)/cost*100:.0f}%")

In [ ]:
answers = {
    "q1": "",  # What is LazyPredict's primary use case and what is it NOT good for?
    "q2": "",  # Why is accuracy a poor metric for churn prediction (hint: class imbalance)?
    "q3": "",  # The top LazyPredict model uses default hyperparameters. What should you do next?
    "q4": "",  # How would you explain AUC-ROC to a non-technical marketing manager?
    "q5": "",  # For churn: would you rather minimize false positives or false negatives? Why?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "LazyPredict Benchmarking"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*